# Pipeline 3 — SQuAD Question Answering

**Owners:** Djem & Sneha  
**Branch:** `develop`

## What this notebook does
Fine-tunes both `bert-base-uncased` and `distilbert-base-uncased` on **SQuAD v1.1** —
an extractive question answering task where the model predicts the start and end position
of the answer span within a passage.

This corresponds to Table 3 in the DistilBERT paper.

Results are saved to `results/` for the final comparison notebook.

## Expected results (from paper Table 3)
| Model | EM | F1 |
|-------|-----|-----|
| BERT-base | 80.8 | 88.5 |
| DistilBERT | 77.7 | 85.8 |

> EM = Exact Match. F1 measures partial overlap between predicted and gold answer.

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────
!pip install -q transformers datasets evaluate accelerate scikit-learn

In [ ]:
import sys
sys.path.append('../src')

import torch
import numpy as np
import collections
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    get_linear_schedule_with_warmup,
)
from datasets import load_dataset
import evaluate as hf_evaluate
from tqdm import tqdm

from utils import print_model_info, save_results, count_parameters, model_size_mb

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
MAX_LENGTH = 384    # standard for SQuAD
DOC_STRIDE = 128    # overlap between chunks for long passages
BATCH_SIZE = 16
EPOCHS     = 2      # paper uses 2 epochs for SQuAD
LR         = 3e-5
SEED       = 42

torch.manual_seed(SEED)

In [ ]:
# ── Load SQuAD v1.1 ────────────────────────────────────────────────────────
# 87,599 train / 10,570 validation question-answer pairs
raw = load_dataset('squad')
print(f"Train: {len(raw['train'])} | Validation: {len(raw['validation'])}")
print(f"\nSample question: {raw['train'][0]['question']}")
print(f"Sample answer:   {raw['train'][0]['answers']['text'][0]}")

In [ ]:
# ── Preprocessing ──────────────────────────────────────────────────────────
# SQuAD requires special preprocessing:
# 1. Tokenize question + context together
# 2. Find the token positions of the answer span
# 3. Handle passages longer than MAX_LENGTH using a sliding window (doc_stride)

def preprocess_train(examples, tokenizer):
    tokenized = tokenizer(
        examples['question'],
        examples['context'],
        truncation='only_second',
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding='max_length',
    )

    sample_map = tokenized.pop('overflow_to_sample_mapping')
    offset_map = tokenized.pop('offset_mapping')

    start_positions, end_positions = [], []

    for i, offsets in enumerate(offset_map):
        sample_idx = sample_map[i]
        answers = examples['answers'][sample_idx]
        sequence_ids = tokenized.sequence_ids(i)

        # Find where context tokens start and end
        ctx_start = sequence_ids.index(1)
        ctx_end   = len(sequence_ids) - 1 - sequence_ids[::-1].index(1)

        if len(answers['answer_start']) == 0:
            start_positions.append(0)
            end_positions.append(0)
            continue

        char_start = answers['answer_start'][0]
        char_end   = char_start + len(answers['text'][0])

        # Check if answer is within this chunk
        if offsets[ctx_start][0] > char_end or offsets[ctx_end][1] < char_start:
            start_positions.append(0)
            end_positions.append(0)
        else:
            tok_start = ctx_start
            while tok_start <= ctx_end and offsets[tok_start][0] <= char_start:
                tok_start += 1
            start_positions.append(tok_start - 1)

            tok_end = ctx_end
            while tok_end >= ctx_start and offsets[tok_end][1] >= char_end:
                tok_end -= 1
            end_positions.append(tok_end + 1)

    tokenized['start_positions'] = start_positions
    tokenized['end_positions']   = end_positions
    return tokenized


def preprocess_val(examples, tokenizer):
    tokenized = tokenizer(
        examples['question'],
        examples['context'],
        truncation='only_second',
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding='max_length',
    )
    tokenized['example_id'] = [
        examples['id'][tokenized['overflow_to_sample_mapping'][i]]
        for i in range(len(tokenized['input_ids']))
    ]
    tokenized.pop('overflow_to_sample_mapping')
    return tokenized

In [ ]:
# ── Training function (QA-specific) ───────────────────────────────────────
def train_qa_model(model, train_loader, device, epochs, lr):
    model.to(device)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )
    history = []

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}'):
            batch = {k: v.to(device) for k, v in batch.items()
                     if k in ('input_ids', 'attention_mask', 'token_type_ids',
                               'start_positions', 'end_positions')}
            outputs = model(**batch)
            loss = outputs.loss
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f'Epoch {epoch+1}: loss={avg_loss:.4f}')
        history.append({'epoch': epoch+1, 'train_loss': round(avg_loss, 4)})

    return history

In [ ]:
# ── Run pipeline ───────────────────────────────────────────────────────────
def run_pipeline(model_name):
    print(f'\n{"="*50}')
    print(f'Model: {model_name} | Task: SQuAD v1.1')
    print(f'{"="*50}')

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Tokenize train set
    train_tokenized = raw['train'].map(
        lambda x: preprocess_train(x, tokenizer),
        batched=True, remove_columns=raw['train'].column_names
    )
    cols = ['input_ids', 'attention_mask', 'start_positions', 'end_positions']
    if 'distilbert' not in model_name:
        cols.append('token_type_ids')
    train_tokenized.set_format('torch', columns=cols)
    train_loader = DataLoader(train_tokenized, batch_size=BATCH_SIZE, shuffle=True)

    model = AutoModelForQuestionAnswering.from_pretrained(model_name)
    print_model_info(model, model_name)

    history = train_qa_model(model, train_loader, DEVICE, epochs=EPOCHS, lr=LR)

    total_params, _ = count_parameters(model)
    results = {
        'model':            model_name,
        'task':             'squad_v1',
        'total_parameters': total_params,
        'model_size_mb':    model_size_mb(model),
        'training_history': history,
        'note':             'Run evaluation separately using HuggingFace evaluate squad metric',
    }

    filename = f"squad_{model_name.replace('/', '_').replace('-', '_')}.json"
    save_results(results, filename)
    return model, tokenizer, results

In [ ]:
# ── Run SQuAD: BERT-base ───────────────────────────────────────────────────
bert_model, bert_tokenizer, bert_squad = run_pipeline('bert-base-uncased')

In [ ]:
# ── Run SQuAD: DistilBERT ──────────────────────────────────────────────────
distil_model, distil_tokenizer, distil_squad = run_pipeline('distilbert-base-uncased')

In [ ]:
# ── Paper vs Ours ─────────────────────────────────────────────────────────
import pandas as pd

rows = [
    {'Model': 'BERT-base (paper)',  'EM': 80.8, 'F1': 88.5},
    {'Model': 'DistilBERT (paper)', 'EM': 77.7, 'F1': 85.8},
    {'Model': 'BERT-base (ours)',   'EM': 'TBD', 'F1': 'TBD'},
    {'Model': 'DistilBERT (ours)',  'EM': 'TBD', 'F1': 'TBD'},
]

# To compute EM/F1, run the HuggingFace squad metric on your predictions:
# metric = evaluate.load('squad')
# results = metric.compute(predictions=predictions, references=references)

print(pd.DataFrame(rows).to_string(index=False))
print('\nSee HuggingFace evaluate docs to compute EM/F1 from model predictions.')